# Module 3 - Scientific Data Manipulation
## 3. Handling Missing Data

Missing values are very common in real-world datasets. In environmental and sensor-based data, they may appear because of communication problems, maintenance, temporary sensor failure, or manual data entry issues.

Handling missing data is not just a technical step. It also requires reasoning. A researcher should ask:

- Why are the values missing?
- How many values are missing?
- Is it acceptable to remove them?
- Should they be filled, and if so, how?

### Learning goals

By the end of this notebook, you should be able to:

- detect missing values,
- measure how much data is missing,
- remove incomplete rows or columns when appropriate,
- fill missing values with simple rules,
- use interpolation for ordered data,
- think critically about which strategy is reasonable.


## 3.1 Example Dataset with Missing Values

We create a small dataset where some measurements are unavailable.


In [1]:
import pandas as pd
import numpy as np


In [2]:
data = {
    "timestamp": pd.date_range("2026-06-01 00:00", periods=8, freq="h"),
    "temperature": [18.2, 18.5, np.nan, 19.0, 19.3, np.nan, 20.1, 20.4],
    "humidity": [65, np.nan, 63, 62, np.nan, 60, 59, 58],
    "co2": [420, 430, 425, np.nan, 440, 445, np.nan, 450]
}

df = pd.DataFrame(data)
df


,timestamp,temperature,humidity,co2
0,2026-06-01 00:00:00,18.2,65.0,420.0
1,2026-06-01 01:00:00,18.5,NaN,430.0
2,2026-06-01 02:00:00,NaN,63.0,425.0
3,2026-06-01 03:00:00,19.0,62.0,NaN
4,2026-06-01 04:00:00,19.3,NaN,440.0
5,2026-06-01 05:00:00,NaN,60.0,445.0
6,2026-06-01 06:00:00,20.1,59.0,NaN
7,2026-06-01 07:00:00,20.4,58.0,450.0


Notice that different columns can have missing values in different rows. This is very common in practice because not all sensors fail at the same time.


## 3.2 Detecting Missing Values

Pandas uses `NaN` to represent many missing numerical values. We can detect them with `isna()` or `isnull()`.


In [3]:
df.isna()


,timestamp,temperature,humidity,co2
0,False,False,False,False
1,False,False,True,False
2,False,True,False,False
3,False,False,False,True
4,False,False,True,False
5,False,True,False,False
6,False,False,False,True
7,False,False,False,False


In [4]:
df.isna().sum()


timestamp      0
temperature    2
humidity       2
co2            2
dtype: int64

In [5]:
(df.isna().mean() * 100).round(1)


timestamp       0.0
temperature    25.0
humidity       25.0
co2            25.0
dtype: float64

These summaries help us quantify missingness. For example, if a column has only one missing value, we may treat it differently than a column with half its values missing.


## 3.3 Looking Only at Incomplete Rows

Sometimes it is helpful to isolate the rows that contain at least one missing value.


In [6]:
incomplete_rows = df[df.isna().any(axis=1)]
incomplete_rows


,timestamp,temperature,humidity,co2
1,2026-06-01 01:00:00,18.5,NaN,430.0
2,2026-06-01 02:00:00,NaN,63.0,425.0
3,2026-06-01 03:00:00,19.0,62.0,NaN
4,2026-06-01 04:00:00,19.3,NaN,440.0
5,2026-06-01 05:00:00,NaN,60.0,445.0
6,2026-06-01 06:00:00,20.1,59.0,NaN


This can help us inspect whether the missing values follow a pattern, such as a short downtime period.


## 3.4 Removing Missing Data

Sometimes it is acceptable to remove rows or columns with missing values. However, this should be done carefully, especially if much data would be lost.


In [7]:
df.dropna()


,timestamp,temperature,humidity,co2
0,2026-06-01 00:00:00,18.2,65.0,420.0
7,2026-06-01 07:00:00,20.4,58.0,450.0


In [8]:
df.dropna(subset=["temperature"])


,timestamp,temperature,humidity,co2
0,2026-06-01 00:00:00,18.2,65.0,420.0
1,2026-06-01 01:00:00,18.5,NaN,430.0
3,2026-06-01 03:00:00,19.0,62.0,NaN
4,2026-06-01 04:00:00,19.3,NaN,440.0
6,2026-06-01 06:00:00,20.1,59.0,NaN
7,2026-06-01 07:00:00,20.4,58.0,450.0


In [9]:
df.dropna(axis=1)


,timestamp
0,2026-06-01 00:00:00
1,2026-06-01 01:00:00
2,2026-06-01 02:00:00
3,2026-06-01 03:00:00
4,2026-06-01 04:00:00
5,2026-06-01 05:00:00
6,2026-06-01 06:00:00
7,2026-06-01 07:00:00


- `dropna()` removes rows with any missing value.
- `dropna(subset=[...])` focuses only on selected columns.
- `dropna(axis=1)` removes columns that contain missing values.

Removing data is simple, but it may reduce the size and quality of the dataset. It is rarely the only option.


## 3.5 Filling Missing Values with Simple Rules

A common strategy is to replace missing values with a constant or with a summary statistic such as the mean or median.


In [10]:
filled_constant = df.fillna(0)
filled_constant


,timestamp,temperature,humidity,co2
0,2026-06-01 00:00:00,18.2,65.0,420.0
1,2026-06-01 01:00:00,18.5,0.0,430.0
2,2026-06-01 02:00:00,0.0,63.0,425.0
3,2026-06-01 03:00:00,19.0,62.0,0.0
4,2026-06-01 04:00:00,19.3,0.0,440.0
5,2026-06-01 05:00:00,0.0,60.0,445.0
6,2026-06-01 06:00:00,20.1,59.0,0.0
7,2026-06-01 07:00:00,20.4,58.0,450.0


In [11]:
filled_mean = df.copy()
filled_mean["temperature"] = filled_mean["temperature"].fillna(filled_mean["temperature"].mean())
filled_mean["humidity"] = filled_mean["humidity"].fillna(filled_mean["humidity"].mean())
filled_mean["co2"] = filled_mean["co2"].fillna(filled_mean["co2"].mean())

filled_mean


,timestamp,temperature,humidity,co2
0,2026-06-01 00:00:00,18.20,65.000000,420.0
1,2026-06-01 01:00:00,18.50,61.166667,430.0
2,2026-06-01 02:00:00,19.25,63.000000,425.0
3,2026-06-01 03:00:00,19.00,62.000000,435.0
4,2026-06-01 04:00:00,19.30,61.166667,440.0
5,2026-06-01 05:00:00,19.25,60.000000,445.0
6,2026-06-01 06:00:00,20.10,59.000000,435.0
7,2026-06-01 07:00:00,20.40,58.000000,450.0


Filling with the mean is easy and sometimes reasonable, but it also reduces variability and may hide structure in the data. It should be used with care.


## 3.6 Forward Fill and Backward Fill

If the data are ordered in time, we can propagate nearby values forward or backward.


In [12]:
forward_filled = df.ffill()
backward_filled = df.bfill()

print("Forward fill:")
display(forward_filled)

print("Backward fill:")
display(backward_filled)


Forward fill:


,timestamp,temperature,humidity,co2
0,2026-06-01 00:00:00,18.2,65.0,420.0
1,2026-06-01 01:00:00,18.5,65.0,430.0
2,2026-06-01 02:00:00,18.5,63.0,425.0
3,2026-06-01 03:00:00,19.0,62.0,425.0
4,2026-06-01 04:00:00,19.3,62.0,440.0
5,2026-06-01 05:00:00,19.3,60.0,445.0
6,2026-06-01 06:00:00,20.1,59.0,445.0
7,2026-06-01 07:00:00,20.4,58.0,450.0


Backward fill:


,timestamp,temperature,humidity,co2
0,2026-06-01 00:00:00,18.2,65.0,420.0
1,2026-06-01 01:00:00,18.5,63.0,430.0
2,2026-06-01 02:00:00,19.0,63.0,425.0
3,2026-06-01 03:00:00,19.0,62.0,440.0
4,2026-06-01 04:00:00,19.3,60.0,440.0
5,2026-06-01 05:00:00,20.1,60.0,445.0
6,2026-06-01 06:00:00,20.1,59.0,450.0
7,2026-06-01 07:00:00,20.4,58.0,450.0


- **Forward fill** copies the previous known value.
- **Backward fill** copies the next known value.

This can be useful when values change slowly, but it may be misleading if the process changes rapidly.


## 3.7 Interpolation

Interpolation estimates missing values from nearby observations. It is often useful in ordered scientific measurements such as time series.


In [13]:
interpolated = df.copy()
interpolated["temperature"] = interpolated["temperature"].interpolate()
interpolated["humidity"] = interpolated["humidity"].interpolate()
interpolated["co2"] = interpolated["co2"].interpolate()

interpolated


,timestamp,temperature,humidity,co2
0,2026-06-01 00:00:00,18.20,65.0,420.0
1,2026-06-01 01:00:00,18.50,64.0,430.0
2,2026-06-01 02:00:00,18.75,63.0,425.0
3,2026-06-01 03:00:00,19.00,62.0,432.5
4,2026-06-01 04:00:00,19.30,61.0,440.0
5,2026-06-01 05:00:00,19.70,60.0,445.0
6,2026-06-01 06:00:00,20.10,59.0,447.5
7,2026-06-01 07:00:00,20.40,58.0,450.0


Interpolation is often more realistic than filling with a single global mean, especially when the values evolve gradually over time.


## 3.8 Comparing Strategies

Different methods produce different results. It is useful to compare them rather than applying one blindly.


In [14]:
comparison = pd.DataFrame({
    "original_temperature": df["temperature"],
    "mean_fill": filled_mean["temperature"],
    "forward_fill": forward_filled["temperature"],
    "interpolation": interpolated["temperature"]
})

comparison


,original_temperature,mean_fill,forward_fill,interpolation
0,18.2,18.20,18.2,18.20
1,18.5,18.50,18.5,18.50
2,NaN,19.25,18.5,18.75
3,19.0,19.00,19.0,19.00
4,19.3,19.30,19.3,19.30
5,NaN,19.25,19.3,19.70
6,20.1,20.10,20.1,20.10
7,20.4,20.40,20.4,20.40


There is no universal best method. The right choice depends on the scientific context and on the purpose of the analysis.


## 3.9 Practical Guidance

A simple decision process for beginners can be:

1. **Measure missingness** first.
2. **Inspect patterns** in the missing values.
3. **Drop rows** only if very little data would be lost.
4. Use **mean/median filling** for simple tabular tasks.
5. Use **forward fill** or **interpolation** when the data are time-ordered and the variable changes gradually.
6. Document what you did, because missing-data handling affects the final results.


## 3.10 Guided Practice

Complete the tasks below.

1. Count missing values in each column.
2. Fill missing temperature values with the mean temperature.
3. Create a version of the data filled with forward fill.
4. Interpolate the `co2` column.


In [15]:
missing_count = ...
temperature_filled = df.copy()
temperature_filled["temperature"] = ...
forward_filled_practice = ...
co2_interpolated = ...

print("Missing values:")
print(missing_count)

print("Temperature filled with mean:")
display(temperature_filled)

print("Forward filled:")
display(forward_filled_practice)

print("Interpolated CO2:")
display(co2_interpolated)


Missing values:
Ellipsis
Temperature filled with mean:


,timestamp,temperature,humidity,co2
0,2026-06-01 00:00:00,Ellipsis,65.0,420.0
1,2026-06-01 01:00:00,Ellipsis,NaN,430.0
2,2026-06-01 02:00:00,Ellipsis,63.0,425.0
3,2026-06-01 03:00:00,Ellipsis,62.0,NaN
4,2026-06-01 04:00:00,Ellipsis,NaN,440.0
5,2026-06-01 05:00:00,Ellipsis,60.0,445.0
6,2026-06-01 06:00:00,Ellipsis,59.0,NaN
7,2026-06-01 07:00:00,Ellipsis,58.0,450.0


Forward filled:


Ellipsis

Interpolated CO2:


Ellipsis

### Suggested solution


In [16]:
missing_count = df.isna().sum()

temperature_filled = df.copy()
temperature_filled["temperature"] = temperature_filled["temperature"].fillna(
    temperature_filled["temperature"].mean()
)

forward_filled_practice = df.ffill()

co2_interpolated = df.copy()
co2_interpolated["co2"] = co2_interpolated["co2"].interpolate()

print("Missing values:")
print(missing_count)

print("Temperature filled with mean:")
display(temperature_filled)

print("Forward filled:")
display(forward_filled_practice)

print("Interpolated CO2:")
display(co2_interpolated)


Missing values:
timestamp      0
temperature    2
humidity       2
co2            2
dtype: int64
Temperature filled with mean:


,timestamp,temperature,humidity,co2
0,2026-06-01 00:00:00,18.20,65.0,420.0
1,2026-06-01 01:00:00,18.50,NaN,430.0
2,2026-06-01 02:00:00,19.25,63.0,425.0
3,2026-06-01 03:00:00,19.00,62.0,NaN
4,2026-06-01 04:00:00,19.30,NaN,440.0
5,2026-06-01 05:00:00,19.25,60.0,445.0
6,2026-06-01 06:00:00,20.10,59.0,NaN
7,2026-06-01 07:00:00,20.40,58.0,450.0


Forward filled:


,timestamp,temperature,humidity,co2
0,2026-06-01 00:00:00,18.2,65.0,420.0
1,2026-06-01 01:00:00,18.5,65.0,430.0
2,2026-06-01 02:00:00,18.5,63.0,425.0
3,2026-06-01 03:00:00,19.0,62.0,425.0
4,2026-06-01 04:00:00,19.3,62.0,440.0
5,2026-06-01 05:00:00,19.3,60.0,445.0
6,2026-06-01 06:00:00,20.1,59.0,445.0
7,2026-06-01 07:00:00,20.4,58.0,450.0


Interpolated CO2:


,timestamp,temperature,humidity,co2
0,2026-06-01 00:00:00,18.2,65.0,420.0
1,2026-06-01 01:00:00,18.5,NaN,430.0
2,2026-06-01 02:00:00,NaN,63.0,425.0
3,2026-06-01 03:00:00,19.0,62.0,432.5
4,2026-06-01 04:00:00,19.3,NaN,440.0
5,2026-06-01 05:00:00,NaN,60.0,445.0
6,2026-06-01 06:00:00,20.1,59.0,447.5
7,2026-06-01 07:00:00,20.4,58.0,450.0


## 3.11 Summary

In this notebook, you learned how to:

- detect missing values,
- quantify missingness,
- inspect incomplete rows,
- remove rows or columns when appropriate,
- fill missing values with simple strategies,
- use forward fill, backward fill, and interpolation,
- choose a strategy more thoughtfully.

In the next notebook, we look at another important data-cleaning issue: **outliers**.
